# Hyperparameter Tuning — Optuna
**Run AFTER data pipeline is ready. Run BEFORE final V1/V2 training.**

This notebook:
1. Runs 30 Optuna trials (2hr budget) on V1 baseline
2. Writes best hyperparams back to `config.yaml`
3. Training then reruns V1 and V2 training with tuned params

In [1]:
# Imports
from src.common_utils import load_config
from src.tuning.study import run_study
from src.tuning.write_best import write_best_to_config
print('Imports OK')

Imports OK


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Quick smoke test — 2 trials to verify everything works
# Change to False to run full study
SMOKE_TEST = False

cfg = load_config(variant='v1')
print(f'Trials: {cfg.tuning.n_trials}')
print(f'Trial epochs: {cfg.tuning.trial_epochs}')
print(f'Timeout: {cfg.tuning.timeout_seconds}s')
print(f'Search space: {dict(cfg.tuning.search_space)}')

Trials: 30
Trial epochs: 15
Timeout: 7200s
Search space: {'lr': [0.0001, 0.01], 'batch_size': [8, 16, 32], 'box_weight': [5.0, 10.0], 'focal_gamma': [0.5, 2.5], 'warmup_epochs': [1, 5]}


In [ ]:
# Run tuning
study = run_study(
    cfg      = cfg,
    n_trials = 2 if SMOKE_TEST else None,   # None = use config value (30)
    timeout  = 300 if SMOKE_TEST else None, # None = use config value (7200s)
)
print(f'\nBest trial: #{study.best_trial.number}')
print(f'Best mAP50: {study.best_value:.4f}')
print(f'Best params: {study.best_trial.params}')

[I 2026-05-02 12:01:11,498] Using an existing study with name 'nvd-yolov9-hp' instead of creating a new one.


12:01:11 | WARNING  | tuning.wandb | optuna W&B integration unavailable — install `optuna-integration[wandb]`.
12:01:11 | INFO     | tuning.study | Starting study 'nvd-yolov9-hp' — 30 trials, 7200s cap, sampler=tpe, pruner=hyperband
12:01:11 | INFO     | tuning.objective | [trial 6] suggested: lr=0.0005611516415334506, batch_size=8, box_weight=5.780093202212183, focal_gamma=0.8119890406724053, warmup_epochs=1
[trainer] YOLOv9 loaded on 0
[trainer] Using existing dataset.
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.45 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10822MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=5.780093202212183, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/project/outp

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



       5/15      9.93G      2.601     0.8924     0.7779         33        640: 100% ━━━━━━━━━━━━ 379/379 2.7it/s 2:18<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 5.0it/s 7.6s0.2s
                   all       1203       4168      0.477      0.298      0.287      0.133
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       7/15       5.6G      2.311     0.8068      0.785         22        640: 100% ━━━━━━━━━━━━ 379/379 2.8it/s 2:18<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 5.1it/s 7.5s0.2s
                   all       1203       4168      0.422      0.264      0.237      0

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.4s0.4s
                   all       1203       4168      0.478      0.346      0.353      0.172

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/15      10.3G      1.905     0.8919      0.774         47        640: 100% ━━━━━━━━━━━━ 379/379 2.6it/s 2:24<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.3s0.4s
                   all       1203       4168      0.437      0.229      0.217      0.114

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/15      6.55G      1.724     0.7563     0.7579         44        640: 100% ━━━━━━━━━━━━ 379/379 2.7it/s 2:21<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.6it/s 7.2s0.4s
                   all   

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



      14/15      10.2G      1.011     0.4352     0.7411         23        640: 100% ━━━━━━━━━━━━ 379/379 2.7it/s 2:18<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.8it/s 6.8s0.4s
                   all       1203       4168      0.475      0.282      0.224      0.126

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      15/15      6.85G     0.9694     0.4188     0.7409         32        640: 100% ━━━━━━━━━━━━ 379/379 2.7it/s 2:18<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.8it/s 6.8s0.4s
                   all       1203       4168      0.483      0.324      0.272       0.16

15 epochs completed in 0.623 hours.
Optimizer stripped from /project/notebooks/runs/detect/outputs/optuna_trials/trial_8/weights/last.pt, 51.6MB
Optimizer stripped from /project/notebooks/runs/detect/outputs/optuna_trials/trial_8/weig

[I 2026-05-02 13:57:39,497] Trial 8 finished with value: 0.32347391998538266 and parameters: {'lr': 0.0002310201887845295, 'batch_size': 32, 'box_weight': 7.159725093210579, 'focal_gamma': 1.0824582803960838, 'warmup_epochs': 4}. Best is trial 8 with value: 0.32347391998538266.


13:57:40 | INFO     | tuning.objective | [trial 9] suggested: lr=0.00019010245319870352, batch_size=32, box_weight=8.925879806965067, focal_gamma=0.8993475643167195, warmup_epochs=3
[trainer] YOLOv9 loaded on 0
[trainer] Using existing dataset.
New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.45 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 10822MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=8.925879806965067, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/project/outputs/yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fractio

In [6]:
# Provide the absolute path to the config file
config_path = "/project/config.yaml"

# Preview what will be written
diff = write_best_to_config(study, config_path=config_path, dry_run=True)

print('\nChanges to config.yaml:')
for k, (old, new) in diff.items():
    print(f'  {k}: {old} → {new}')

15:26:17 | INFO     | tuning.write_best | [dry-run] would change:
15:26:17 | INFO     | tuning.write_best |    training.lr: 0.001 → 0.00019010245319870352
15:26:17 | INFO     | tuning.write_best |    training.batch_size: 16 → 32
15:26:17 | INFO     | tuning.write_best |    loss.box_weight: 7.5 → 8.925879806965067
15:26:17 | INFO     | tuning.write_best |    loss.focal_gamma: 1.5 → 0.8993475643167195

Changes to config.yaml:
  training.lr: 0.001 → 0.00019010245319870352
  training.batch_size: 16 → 32
  loss.box_weight: 7.5 → 8.925879806965067
  loss.focal_gamma: 1.5 → 0.8993475643167195


In [9]:
# Write best params to config.yaml
# Skip if smoke test
if not SMOKE_TEST:
    write_best_to_config(study, config_path="/project/config.yaml")
    print('config.yaml updated!')
else:
    print('Smoke test — skipping config.yaml write.')

15:31:44 | INFO     | tuning.write_best | Backup saved to /project/config.yaml.bak-20260502-153144
15:31:44 | INFO     | tuning.write_best | Patched /project/config.yaml with best trial #9
15:31:44 | INFO     | tuning.write_best |    training.lr: 0.001 → 0.00019010245319870352
15:31:44 | INFO     | tuning.write_best |    training.batch_size: 16 → 32
15:31:44 | INFO     | tuning.write_best |    loss.box_weight: 7.5 → 8.925879806965067
15:31:44 | INFO     | tuning.write_best |    loss.focal_gamma: 1.5 → 0.8993475643167195
config.yaml updated!


In [10]:
# Plot results
import optuna
import matplotlib.pyplot as plt

fig = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.savefig('outputs/results/tuning_history.png')
print('Saved: outputs/results/tuning_history.png')

fig2 = optuna.visualization.matplotlib.plot_param_importances(study)
plt.savefig('outputs/results/tuning_param_importance.png')
print('Saved: outputs/results/tuning_param_importance.png')

Saved: outputs/results/tuning_history.png
Saved: outputs/results/tuning_param_importance.png
